In [ ]:
%reload_ext autoreload
%autoreload 2

# Agent-Based Task Execution

This notebook demonstrates how to use the `Agent` pipeline from **[OnPrem.LLM](https://github.com/amaiya/onprem)** to create autonomous agents that can execute complex tasks using a variety of tools.

AgentExecutor works with any LiteLLM-supported model that supports tool-calling:

- Cloud: openai/gpt-5.2-codex, anthropic/claude-sonnet-4-5, gemini/gemini-1.5-pro
- Local: Ollama (ollama/llama3.1), vLLM (hosted_vllm/<model>), llama.cpp (use OpenAI interface)

For **llama.cpp**:  Use `openai/gpt-oss-120b` as model parameter and then set env variable `OPENAI_API_BASE=http://localhost:<port>/v1`.

## The AgentExecutor

The `AgentExecutor` allows you to launch AI agents to solve various tasks using both cloud and local models. We will use **anthropic/claude-sonnet-4-5** (cloud) and **glm-4.7-flash** (local) for these examples.

By default, the `AgentExecutor` has access to 9 built-in tools.  You remove access to built-in-tools as necessary.  You can optionally give the agent access to **custom tools**, as we'll illustrate below.

The `AgentExecutor` is implemented using our coding agent, PatchPal, which you'll need to install: `pip install patchpal`.

In [ ]:
# | notest

from onprem.pipelines import AgentExecutor

In [ ]:
# | notest

AgentExecutor.print_default_tools()

AgentExecutor Default Tools

These tools are used by default when enabled_tools=None:

   1. read_file       - Read complete file contents
   2. read_lines      - Read specific line ranges from files
   3. edit_file       - Edit files via find/replace
   4. write_file      - Write complete file contents
   5. grep            - Search for patterns in files
   6. find            - Find files by glob pattern
   7. run_shell       - Execute shell commands
   8. web_search      - Search the web for information
   9. web_fetch       - Fetch and read content from URLs

Customization Examples:

# Use defaults (all tools including shell):
executor = AgentExecutor(model='anthropic/claude-sonnet-4-5')

# Defaults but no shell access (safer):
executor = AgentExecutor(
    model='anthropic/claude-sonnet-4-5',
    disable_shell=True
)

# Minimal tools:
executor = AgentExecutor(
    model='anthropic/claude-sonnet-4-5',
    enabled_tools=['read_file', 'write_file']
)

# Web research only:
executor = A

## Examples

Let's run through some examples for different scenarios.

### Basic Calculator Example

In this introductory example, we will launch an agent to build a **calculator module in Python**.

By default, the agent will operate within the working_directory you specify (or the current folder if not working directory is specified).  The agent cannot read or write outside the working directory.

**Important Note:**:  If the agent has access to the `run_shell` tool, it can potentially read or modify files outside of the working directory (e.g., auto-generating and executing a Python script that writes files outside of working directory).  For these reasons, you can either supply the `disable_shell=True` to remove shell access or `sandbox=True`, which runs the agent in an ephemeral container.  We will cover sandboxed execution later. 

We will run this example without either, as we are running this notebook within a VM.

In [ ]:
# | notest

executor = AgentExecutor(
     model='anthropic/claude-sonnet-4-5', # assumes ANTHROPIC_API_KEY is already set as environment variable
     max_iterations=10
 )

result = executor.run(
     task="""
     Create a simple Python calculator module with the following:
     - calculator.py with add, subtract, multiply, divide functions
     - test_calculator.py with pytest tests
     - All tests must pass
     """,
     working_dir='./calculator_project'
)
print(result)


✈️  PatchPal Autopilot Mode Starting
Prompt: 
     Create a simple Python calculator module with the following:
     - calculator.py with add, su...
Completion promise: 'COMPLETE'
Max iterations: 10
Model: anthropic/claude-sonnet-4-5
Working directory: /home/amaiya/projects/ghub/onprem/nbs/calculator_project
🔒 File access restricted to working directory


🔄 Autopilot Iteration 1/10

🤔 Thinking...

I'll inspect the repository to see what's already present, then add the         
calculator module and tests. I'll run the test suite and fix any issues until   
all tests pass. I'll show test run output and then finish with the required     
completion tag. I'll list files first.                                          

📂 Finding files: **/*
🤔 Thinking...
📖 Reading: task_prompt.md
🤔 Thinking...

I'll create calculator.py and test_calculator.py, then run the pytest suite to  
confirm all tests pass. I'll write the files now.                               

📝 Patching: calculator.py
🤔 Think

### Web Research Agent

This example is a web research agent that only has access to the following tools: `web_search`, `web_fetch`, `write_file`.

In [ ]:
# | notest

prompt = """
Research the latest developments in quantum computing in 2026.
Create a markdown report called 'quantum_computing_2026.md' with:
- Executive summary
- Key breakthroughs
- Major companies/institutions involved
- Potential applications
- Sources and references
"""

In [ ]:
# | notest

# using Claude Sonnet 4.5 on AWS Bedrock here
AWS_GOVCLOUD_ARN = "arn:aws-us-gov:bedrock:us-gov-east-1:393736811581:inference-profile/us-gov.anthropic.claude-sonnet-4-5-20250929-v1:0"
executor = AgentExecutor(
     model=AWS_GOVCLOUD_ARN,
     max_iterations=10,
    enabled_tools=["web_search", "web_fetch", "write_file"],
 )

result = executor.run(
     task=prompt,
     working_dir='./quantum_report'
)


✈️  PatchPal Autopilot Mode Starting
Prompt: 
Research the latest developments in quantum computing in 2026.
Create a markdown report called 'qua...
Completion promise: 'COMPLETE'
Max iterations: 10
Model: bedrock/arn:aws-us-gov:bedrock:us-gov-east-1:393736811581:inference-profile/us-gov.anthropic.claude-sonnet-4-5-20250929-v1:0
Working directory: /home/amaiya/projects/ghub/onprem/nbs/quantum_report
🔒 File access restricted to working directory


🔄 Autopilot Iteration 1/10

🤔 Thinking...

I'll research the latest developments in quantum computing in 2026 and create a 
comprehensive markdown report.                                                  

🌐 Searching web: quantum computing breakthroughs 2026
🌐 Searching web: quantum computing companies developments 2026
🌐 Searching web: quantum computing applications 2026
🤔 Thinking...

Now let me fetch detailed information from some of the key sources:             

🌐 Fetching: https://www.forbes.com/sites/bernardmarr/2025/12/11/7-quantum-c

In [ ]:
!pwd

/home/amaiya/projects/ghub/onprem/nbs


In [ ]:
# | notest

from IPython.display import display, HTML
from markdown import markdown

with open('./quantum_report/quantum_computing_2026.md', 'r') as f:
    lines = [f.readline() for _ in range(20)]
    content = ''.join(lines)

html_content = markdown(content)

display(HTML(f"""
 <div style="
     border-left: 4px solid #4CAF50;
     padding: 15px 20px;
     margin: 15px 0;
     background-color: #f0f7f4;
     border-radius: 4px;
     box-shadow: 0 2px 4px rgba(0,0,0,0.1);
 ">
     <h4 style="margin-top: 0; color: #2e7d32;">📄 First 20 lines of markdown report from agent:</h4>
     {html_content}
 </div>
 """))

### Local Models

The `AgentExecutor` supports local models.  By default, it will assume the local model supports native function-calling (e.g., gpt-oss-120b).  If you use a local model that does **not** have good native support for function-calling (a.k.a. tool-calling), you can change the agent_type to `react`. In this example, we will use `llama3.1:8b`.

**Note:** The default context window length in Ollama is typically too small for agentic  workflows.  Depending on the model and task, we recommend inreasing to at least 8192.  Reasoning models like gpt-oss:120b may require 32K or 64K.  

```sh
OLLAMA_CONTEXT_LENGTH=32000 ollama serve
```

In [ ]:
# | notest

executor = AgentExecutor(
     model='ollama_chat/glm-4.7-flash:q4_K_M',
     enabled_tools=["web_fetch", "write_file"],
     max_iterations=10
 )

result = executor.run(
     task="What is the highest level of education of the person listed on this page: https://arun.maiya.net? Write answer in answer.txt.",
     working_dir='./extraction_example'
)


✈️  PatchPal Autopilot Mode Starting
Prompt: What is the highest level of education of the person listed on this page: https://arun.maiya.net? Wr...
Completion promise: 'COMPLETE'
Max iterations: 10
Model: ollama_chat/glm-4.7-flash:q4_K_M
Working directory: /home/amaiya/projects/ghub/onprem/nbs/extraction_example
🔒 File access restricted to working directory


🔄 Autopilot Iteration 1/10

🤔 Thinking...

I'll fetch the webpage to find the highest level of education listed for the    
person.                                                                         

🌐 Fetching: https://arun.maiya.net
🤔 Thinking...

I found the education information on the page. The person states: "I completed a
Ph.D. in Computer Science at the Laboratory for Computational Population        
Biology..." The highest level of education is a Ph.D.                           

I'll write this to answer.txt.                                                  

📝 Patching: answer.txt
🤔 Thinking...

📝 Agent Response

In [ ]:
# | notest

!ls ./extraction_example/

answer.txt


In [ ]:
# | notest

!cat ./extraction_example/answer.txt

Ph.D. in Computer Science

### Sandboxed Execution

For enhanced security and isolation, set sandbox=True to run the agent in an ephemeral Docker/Podman container. This is useful when working with
untrusted code, needing resource limits, or wanting to protect your file system from accidental modifications. The tradeoff is ~5-10 seconds of
container startup overhead, but you gain a clean, reproducible environment that's automatically cleaned up after execution.

> Prerequisites: Requires Docker or Podman installed. See docker.com or podman.io.

This examples launches the agent in an ephemeral Docker container.

In [ ]:
# | notest

prompt = """
Create a Python script that:
1. Generates sample sales data for 12 months (random)
2. Calculates total sales, average, min, max
3. Creates a matplotlib bar chart saved as 'sales_chart.png'
4. Writes a summary report to 'sales_analysis.txt'
"""

In [ ]:
# | notest

executor = AgentExecutor(
     model='anthropic/claude-sonnet-4-5',
     max_iterations=10,
    sandbox=True
 )

result = executor.run(
     task=prompt,
     working_dir='./data_analysis'
)

Unable to find image 'python:3.11-slim' locally
3.11-slim: Pulling from library/python
206356c42440: Pulling fs layer
206709444425: Pulling fs layer
bbf418262cbe: Pulling fs layer
43272e9c9d4b: Pulling fs layer
43272e9c9d4b: Waiting
206709444425: Verifying Checksum
206709444425: Download complete
43272e9c9d4b: Download complete
bbf418262cbe: Verifying Checksum
bbf418262cbe: Download complete
206356c42440: Download complete
206356c42440: Pull complete
206709444425: Pull complete
bbf418262cbe: Pull complete
43272e9c9d4b: Pull complete
Digest: sha256:d6e4d224f70f9e0172a06a3a2eba2f768eb146811a349278b38fff3a36463b47
Status: Downloaded newer image for python:3.11-slim

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

✈️  PatchPal Autopilot Mode Starting
Prompt: 
Create a Python script that:
1. Generates sample sales data for 12 months (random)
2. Calculates to...
Completion promise: 'COMPLETE'
Max iterations: 10
Model: anthropic/c

In [ ]:
# | notest
!ls ./data_analysis/

generate_sales_report.py


In [ ]:
# | notest

from IPython.display import display, HTML, Code

display(HTML("""
 <div style="
     border-left: 4px solid #4CAF50;
     padding: 15px 20px;
     margin: 15px 0;
     background-color: #f0f7f4;
     border-radius: 4px;
     box-shadow: 0 2px 4px rgba(0,0,0,0.1);
 ">
     <h4 style="margin-top: 0; color: #2e7d32;">📜 generate_sales_report.py</h4>
"""))

display(Code(filename='./data_analysis/generate_sales_report.py', language='python'))

display(HTML('</div>'))

#!/usr/bin/env python3
"""
Generate sample sales data for 12 months, compute stats, save chart and summary.
Creates:
 - sales_chart.png
 - sales_analysis.txt
"""
import random
import statistics
import os
import matplotlib
matplotlib.use('Agg')  # use non-interactive backend suitable for scripts
import matplotlib.pyplot as plt


def generate_sales(months=12, low=5000, high=20000):
    """Return a list of `months` random sales values rounded to 2 decimals."""
    return [round(random.uniform(low, high), 2) for _ in range(months)]


def analyze(sales, months_names):
    total = round(sum(sales), 2)
    avg = round(statistics.mean(sales), 2)
    minimum = min(sales)
    maximum = max(sales)
    min_month = months_names[sales.index(minimum)]
    max_month = months_names[sales.index(maximum)]
    return dict(total=total, average=avg, min=minimum, max=maximum, min_month=min_month, max_month=max_month)


def save_chart(months_names, sales, filename='sales_chart.png'):
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.bar(months_names, sales, color='skyblue')
    ax.set_title('Monthly Sales')
    ax.set_xlabel('Month')
    ax.set_ylabel('Sales ($)')
    ax.set_xticks(range(len(months_names)))
    ax.set_xticklabels(months_names, rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(filename)
    plt.close(fig)


def write_report(filename, sales, analysis, months_names):
    lines = []
    lines.append('Sales Analysis Report')
    lines.append('=' * 40)
    lines.append(f"Total sales: ${analysis['total']}")
    lines.append(f"Average monthly sales: ${analysis['average']}")
    lines.append(f"Minimum sales: ${analysis['min']} ({analysis['min_month']})")
    lines.append(f"Maximum sales: ${analysis['max']} ({analysis['max_month']})")
    lines.append('')
    lines.append('Monthly breakdown:')
    for m, s in zip(months_names, sales):
        lines.append(f"{m}: ${s}")
    with open(filename, 'w') as f:
        f.write('\n'.join(lines))
    return '\n'.join(lines)


def main():
    months_names = ['January', 'February', 'March', 'April', 'May', 'June',
                    'July', 'August', 'September', 'October', 'November', 'December']
    # Generate sample data
    sales = generate_sales(len(months_names))

    # Analyze
    analysis = analyze(sales, months_names)

    # Save chart and report
    save_chart(months_names, sales, filename='sales_chart.png')
    write_report('sales_analysis.txt', sales, analysis, months_names)

    print('Generated sales_chart.png and sales_analysis.txt')


if __name__ == '__main__':
    main()

 ### Local Models + Sandbox: Networking Setup

 Local models (Ollama, llama.cpp) on localhost need container networking configured:

 - **Linux/WSL2**: Supply `network='host'` to `AgentExecutor`. (WSL2: make sure to enable mirrored networking in `.wslconfig`.)
 - **macOS/Windows**: Set `OLLAMA_API_BASE='http://host.docker.internal:11434'` (Docker) or `http://host.containers.internal:11434` (Podman)

### Custom Tools

You can give the agent custom tools by simply defining them as Python functions or callables.

In this example, we'll build a financial analysis agent with custom tools.

Let's first definte the tools, which are based on yfinance. 

> `pip install yfinance`

#### Step 1: Define the  custom tools as Python functions

In [ ]:
# | notest

from typing import List, Dict
from datetime import datetime, timedelta

# Define custom financial tools


def get_current_stock_price(ticker: str) -> Dict[str, float]:
    """
    Fetch current/live stock price for a given ticker.
    
    Args:
        ticker: Stock ticker symbol (e.g., 'AAPL', 'GOOGL')
     Returns:
        Dictionary with current price and related info
    """
    try:
        import yfinance as yf
        from datetime import datetime
        stock = yf.Ticker(ticker)
        info = stock.info

        # Get current price (live during market hours, last close otherwise)
        current_price = info.get('currentPrice') or info.get('regularMarketPrice')

        return {
            "ticker": ticker.upper(),
            "current_price": round(current_price, 2),
            "market_state": info.get('marketState', 'unknown'),  # 'REGULAR', 'CLOSED', etc.
            "timestamp": datetime.now().isoformat()
        }
    except Exception as e:
        return {"error": str(e)}


def calculate_return_percentage(purchase_price: float, current_price: float) -> float:
     """
     Calculate percentage return on investment.
    
     Args:
         purchase_price: Original purchase price per share
         current_price: Current market price per share
    
     Returns:
         Percentage return (positive for gains, negative for losses)
     """
     if purchase_price == 0:
         return 0.0
     return round(((current_price - purchase_price) / purchase_price) * 100, 2)

def analyze_volatility(ticker: str, days: int = 30) -> Dict[str, float]:
     """
     Calculate stock volatility metrics over a period.
    
     Args:
         ticker: Stock ticker symbol
         days: Number of days to analyze
    
     Returns:
         Dictionary with volatility metrics
     """
     try:
         import yfinance as yf
         stock = yf.Ticker(ticker)
         end_date = datetime.now()
         start_date = end_date - timedelta(days=days + 10)
         hist = stock.history(start=start_date, end=end_date)
    
         if hist.empty or len(hist) < 2:
             return {"error": f"Insufficient data for {ticker}"}
    
         daily_changes = hist['Close'].pct_change() * 100
    
         return {
             "ticker": ticker.upper(),
             "period_days": len(hist),
             "avg_daily_change": round(abs(daily_changes.mean()), 2),
             "max_increase": round(daily_changes.max(), 2),
             "max_decrease": round(daily_changes.min(), 2),
             "std_deviation": round(daily_changes.std(), 2)
         }
     except Exception as e:
         return {"error": str(e)}

#### Step 2: Launch the agent within with access to the custom tools

In [ ]:
# | notest

# Create agent with custom tools
from onprem.pipelines.agent import AgentExecutor

executor = AgentExecutor(
 model='anthropic/claude-sonnet-4-5',
 custom_tools=[calculate_return_percentage, analyze_volatility, get_current_stock_price],
 enabled_tools=['write_file'],
 verbose=True
)

# Task: Analyze Apple and Microsoft stock
task = """
Create a stock analysis report for Apple (AAPL) and Microsoft (MSFT):

1. Get current stock prices for both companies
2. Analyze volatility for both over the last 30 days
3. If I bought AAPL at $150 and it's now at current price, calculate my return percentage
4. Create a markdown report comparing the two stocks

Save the report to 'stock_analysis.md'
"""

result = executor.run(task, working_dir='./financial_workspace')

✓ Wrote custom tool 'calculate_return_percentage' to .patchpal/tools/calculate_return_percentage.py
✓ Wrote custom tool 'analyze_volatility' to .patchpal/tools/analyze_volatility.py
✓ Wrote custom tool 'get_current_stock_price' to .patchpal/tools/get_current_stock_price.py

✈️  PatchPal Autopilot Mode Starting
Prompt: 
Create a stock analysis report for Apple (AAPL) and Microsoft (MSFT):

1. Get current stock prices ...
Completion promise: 'COMPLETE'
Max iterations: 50
Model: anthropic/claude-sonnet-4-5
Working directory: /home/amaiya/projects/ghub/onprem/nbs/financial_workspace
🔒 File access restricted to working directory
🔧 Custom tools: analyze_volatility, calculate_return_percentage, get_current_stock_price


🔄 Autopilot Iteration 1/50

🤔 Thinking...

I'll fetch current prices and 30-day volatility metrics for AAPL and MSFT in    
parallel. After that I'll compute your AAPL return (bought at $150) and write a 
markdown report to stock_analysis.md. Proceeding to fetch the data now. 

In [ ]:
# | notest

from IPython.display import display, HTML
from markdown import markdown

with open('./financial_workspace/stock_analysis.md', 'r') as f:
    lines = [f.readline() for _ in range(16)]
    content = ''.join(lines)

html_content = markdown(content)

display(HTML(f"""
 <div style="
     border-left: 4px solid #4CAF50;
     padding: 15px 20px;
     margin: 15px 0;
     background-color: #f0f7f4;
     border-radius: 4px;
     box-shadow: 0 2px 4px rgba(0,0,0,0.1);
 ">
     <h4 style="margin-top: 0; color: #2e7d32;">📄 First 16 lines of markdown report from agent:</h4>
     {html_content}
 </div>
 """))